In [51]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import cv2
import os
import shutil
import sys
import csv

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
NVIDIA GeForce RTX 4050 Laptop GPU


In [52]:
detect_model = YOLO("models/11L3.pt")
pose_model = YOLO("yolo11l-pose")

video_input_directory = Path("video/input")
video_output_directory = Path("video/output")
log_directory = Path("video/logs")
gt_directory = Path("video/gt")

video_input_directory.mkdir(parents=True, exist_ok=True)
video_output_directory.mkdir(parents=True, exist_ok=True)
log_directory.mkdir(parents=True, exist_ok=True)
gt_directory.mkdir(parents=True, exist_ok=True)

detect_classes = [k for k, v in detect_model.names.items() if v != "person"]
pose_classes = [k for k, v in pose_model.names.items() if v == "person"]
custom_tracker = "trackers/custom_botsort.yaml"

bar_length = 40

video_stride = 5
ioa_thres = 0.5

In [53]:
for item in video_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

for item in log_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [54]:
for video_index, video in enumerate(video_input_directory.glob("*.mp4")):
    print(video.stem)

vid1


In [55]:
for video_index, video in enumerate(video_input_directory.glob("*.mp4")):

    video_name = video.stem
    print(f"Video: {video_name}")

    cap = cv2.VideoCapture(str(video))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1

    adjusted_fps = (fps / video_stride)

    output_video = str(video_output_directory / f"{video_name}_output.mp4")
    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"avc1"),
        fps,
        (width, height)
    )

    hoi_output_log = str(log_directory / f"{video_name}_interactions.csv")
    frame_count = 0
    
    with open(hoi_output_log, "w") as hoi_log:
        try:

            current_frame = None
            
            # Main Loop
            while cap.isOpened():

                success, frame = cap.read()
                if not success:
                    break

                if frame_count % video_stride == 0:

                    current_frame = frame

                    est_time_seconds = int(frame_count / fps)
                    minutes, seconds = divmod(est_time_seconds, 60)
                    base_log = f"{frame_count:05},{minutes:02}:{seconds:02},"

                    filled = int(bar_length * (frame_count + 1) // total_frames)
                    bar = '█' * filled + '-' * (bar_length - filled)
                    frame_percent = ((frame_count + 1) / total_frames) * 100
                    print(f"\r\033[KProcessing Frame: {frame_count + 1}/{total_frames} [{bar}] {frame_percent:.2f}%", end="", flush=True)
                    
                    detect_results = detect_model.track(
                        device=0,
                        source=current_frame,
                        tracker=custom_tracker,
                        persist=True,
                        conf=0.5,
                        verbose=False,
                        classes=detect_classes
                    )

                    pose_results = pose_model.track(
                        device=0,
                        source=current_frame,
                        tracker=custom_tracker,
                        persist=True,
                        conf=0.5,
                        verbose=False,
                        classes=pose_classes
                    )

                    d_result = detect_results[0]
                    p_result = pose_results[0]
                    
                    # Plot the pose keypoints to the frame
                    current_frame = p_result.plot(boxes=False)
                    d_boxes = d_result.boxes
                    p_boxes = p_result.boxes
                    
                    # Plot the object rects to the frame
                    if len(d_boxes) > 0:

                        d_xyxys = d_boxes.xyxy.cpu().numpy()
                        d_clss = d_boxes.cls.cpu().numpy().astype(int)
                        d_confs = d_boxes.conf.cpu().numpy()
                        d_track_ids = d_boxes.id.cpu().numpy().astype(int) if d_boxes.id is not None else [0] * len(d_boxes)

                        for i, (x1, y1, x2, y2) in enumerate(d_xyxys):

                            d_rgb = (255, 0, 0)
                            cv2.rectangle(
                                current_frame,
                                (int(x1), int(y1)),
                                (int(x2), int(y2)),
                                d_rgb,
                                2
                            )

                            label = f"ID:{d_track_ids[i]} CLS:{detect_model.names[d_clss[i]]} CONF:{round(float(d_confs[i]), 2)}"
                            cv2.putText(
                                current_frame,
                                label,
                                (int(x1), int(y1)-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                                d_rgb,
                                2
                            )

                    # Plot the pose rects for humans to the frame
                    if len(p_boxes) > 0:

                        p_xyxys = p_boxes.xyxy.cpu().numpy()
                        p_confs = p_boxes.conf.cpu().numpy()
                        p_track_ids = p_boxes.id.cpu().numpy().astype(int) if p_boxes.id is not None else [0] * len(p_boxes)
                        kpts = p_result.keypoints.xy.cpu().numpy() if p_result.keypoints is not None else None

                        for i, (x1, y1, x2, y2) in enumerate(p_xyxys):

                            p_rgb = (0, 0, 255)
                            cv2.rectangle(
                                current_frame,
                                (int(x1), int(y1)),
                                (int(x2), int(y2)),
                                p_rgb,
                                2
                            )

                            label = f"ID:{p_track_ids[i]} CLS:person CONF:{round(float(p_confs[i]), 2)}"
                            cv2.putText(
                                current_frame,
                                label,
                                (int(x1), int(y1)-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                                p_rgb,
                                2
                            )

                    # All the math calculations etc
                    if len(d_boxes) > 0:

                        xi1 = np.maximum(p_xyxys[:, None, 0], d_xyxys[None, :, 0])
                        yi1 = np.maximum(p_xyxys[:, None, 1], d_xyxys[None, :, 1])
                        xi2 = np.minimum(p_xyxys[:, None, 2], d_xyxys[None, :, 2])
                        yi2 = np.minimum(p_xyxys[:, None, 3], d_xyxys[None, :, 3])
                        
                        inter_area = np.maximum(0, xi2 - xi1) * np.maximum(0, yi2 - yi1)
                        obj_area = (d_xyxys[:, 2] - d_xyxys[:, 0]) * (d_xyxys[:, 3] - d_xyxys[:, 1])
                        
                        ioa = np.zeros_like(inter_area)
                        valid_obj = obj_area > 0
                        ioa[:, valid_obj] = inter_area[:, valid_obj] / obj_area[valid_obj]

                        p_indices, d_indices = np.where(ioa >= ioa_thres)

                        for p_i, d_i in zip(p_indices, d_indices):

                            if kpts is not None and len(kpts) > p_i:
                                person_kpts = kpts[p_i]
                                dx1, dy1, dx2, dy2 = d_xyxys[d_i]
                                in_box = (person_kpts[..., 0] >= dx1) & (person_kpts[..., 0] <= dx2) & (person_kpts[..., 1] >= dy1) & (person_kpts[..., 1] <= dy2)
                                
                                if in_box.any():
                                    hoi_log.write(base_log + f"{p_track_ids[p_i]},{d_track_ids[d_i]},{detect_model.names[d_clss[d_i]]}\n")

                output_frame = current_frame.copy()
                frame_label = f"Frame {frame_count}"
                cv2.putText(
                    output_frame,
                    frame_label,
                    (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255),
                    3
                )

                writer.write(output_frame)

                frame_count += 1
                
        except Exception as e:

            if writer is not None:
                writer.release()

            print(f"\n\nCrash detected!\n{e}")
            sys.exit(1)

        finally:
            cap.release()
            writer.release()

    print(f"\nCompleted {video}\n")
    
print("All Done")

Video: vid1
Processing Frame: 3921/3921 [████████████████████████████████████████] 100.00%
Completed video\input\vid1.mp4

All Done


In [62]:
frame_tolerance = 150

for csv_index, csv_file in enumerate(log_directory.glob("*.csv")):

    csv_name = csv_file.stem

    summary = {}

    with open(csv_file, 'r') as log_file:

        for line in log_file:

            if not line.strip():
                continue
            
            cols = line.strip().split(',')
            frame, time_str, human_id, object_id, object_class = int(cols[0]), cols[1], cols[2], cols[3], cols[4]

            key = (human_id, object_id, object_class)

            if key not in summary:
                summary[key] = [[frame, frame, time_str, time_str]]
            
            elif frame - summary[key][-1][1] <= frame_tolerance:
                summary[key][-1][1] = frame
                summary[key][-1][3] = time_str

            else:
                summary[key].append([frame, frame, time_str, time_str])

    with open(csv_file.with_name(f"{csv_name}_summary.csv"), 'w') as summary_log:
        
        for (h, o, c), events in summary.items():
            for start_f, end_f, start_t, end_t in events:
                duration_sec = (int(end_t.split(':')[0]) * 60 + int(end_t.split(':')[1])) - (int(start_t.split(':')[0]) * 60 + int(start_t.split(':')[1]))
                duration_frames = end_f - start_f
                line_to_write = f"{h},{o},{c},{start_f},{end_f},{duration_frames},{start_t},{end_t},{duration_sec}\n"
                if duration_frames >= 30:
                    summary_log.write(line_to_write)